# React — HTTP and APIs

> **Reminder of the three places.** This notebook runs plain JavaScript on the Deno kernel.
> `playground/` is where you see real React behave. Mini-projects are where you build.
> Blocks are labelled **Runnable — plain JS**, **The React API**, **In the playground** or
> **In your project**.
>
> **You already know `fetch`.** The JavaScript course taught it properly — `response.ok`,
> status codes, `.json()`, `try`/`catch`, and the fact that a 404 is not an error as far as
> `fetch` is concerned. None of that is re-taught here. This topic is about what changes when
> the request lives inside a component.
>
> Every runnable cell in this topic works **offline**: they run against a local mock response.
> The real network request is in the playground, where you can watch it in the Network tab.

## LESSON 45 — Getting data into a component

Here is what you already know, from the JavaScript course:

```js
const response = await fetch("https://example.com/users");

if (!response.ok) {
  throw new Error(`Request failed with status ${response.status}`);
}

const users = await response.json();
```

Every line of that is still correct. Three things change when it moves inside a component,
and all three follow from lessons you have already done.

### 1. Where the call goes

Not at the top of the module, and not in the component body. Fetching is caused by **the
component being on screen** — nobody clicked anything — which is LESSON 39's definition of an
Effect. And because it is async and can be superseded, it needs LESSON 43's shape:

```jsx
useEffect(() => {
  let ignore = false;

  async function load() {
    const response = await fetch(url);
    if (!response.ok) throw new Error(`Request failed with status ${response.status}`);
    const data = await response.json();
    if (!ignore) setUsers(data);
  }

  load();
  return () => { ignore = true; };
}, [url]);
```

There is nothing new in that block. It is L39 (caused by rendering), L40 (`[url]`), L41
(cleanup), L43 (inner async function, `ignore`) and your existing `fetch` knowledge, assembled.

### 2. The result has to become state

In a script, `const users = await fetch(...)` is the end of the job. In a component it is the
*middle* of it: a local variable disappears when the function returns, and nothing would tell
React to render again (LESSON 25's two reasons, unchanged).

So the response goes into state, and the state is what you render. The fetch does not put
anything on screen — it changes state, and the state change causes the render that does.

### 3. The first render has no data

This is the part that surprises people, and it is worth seeing rather than being told.
Measured in the playground, one click of "load users":

```text
   render — loading:false users:null error:no      <- nothing yet
   request — https://jsonplaceholder.typicode.com/users
   render — loading:true users:null error:no       <- still nothing
   ok — 10 users
   render — loading:false users:10 error:no        <- now there is data
```

Your component runs **before** the data exists, every single time. There is no arrangement of
code that avoids this: the request cannot start until the component has rendered at least once,
because that is when the Effect runs (LESSON 39).

So `users` starts as `null` — or `[]`, your choice — and your JSX must be able to render that
state without crashing. `users.map(...)` on `null` throws, and it will throw on the very first
render of every component that forgets. What the screen should show during that moment is
LESSON 46.

### `response.ok` matters more here, not less

You know that `fetch` does not throw on 404 or 500. In a script, forgetting `response.ok`
gives you a confusing crash a line later. In a component it is worse: you call
`response.json()` on the error body, get a perfectly valid object that is not your data, put
it in state, and render it. No error is thrown, your error branch never runs, and the screen
shows something wrong with complete confidence.

Measured, with the ok-check in place and a deliberately broken URL:

```text
   request — https://jsonplaceholder.typicode.com/no-such-collection
   error — Request failed with status 404
   render — loading:false users:null error:yes
```

The check is what turns a "successful" 404 into something your UI can react to.

### Key Notes

- The call belongs in an **Effect** — it is caused by rendering, not by a click.
- The response must become **state**, or there is nothing to render and nothing to trigger a
  render.
- Your component **always renders once with no data**. The JSX must survive that.
- `response.ok` is what stops an error body being rendered as if it were your data.

### Example

**Runnable — plain JS, offline.** The part of this worth extracting is the bit that turns a
response into something your component can use. That is ordinary JavaScript, it is the same
function you will put in a services file in LESSON 48, and it needs no network to test —
which is exactly why it is worth having on its own.

In [ ]:
// Turn a response into a plain result object. No React, no network.
async function l45toResult(response) {
  if (!response.ok) {
    return { ok: false, error: `Request failed with status ${response.status}` };
  }
  const data = await response.json();
  return { ok: true, data };
}

// A stand-in for a real Response. The real one has these too.
function l45mockResponse(status, body) {
  return {
    ok: status >= 200 && status < 300,
    status,
    json: async () => body,
  };
}

console.log("200 ->", JSON.stringify(await l45toResult(l45mockResponse(200, [{ id: 1 }]))));
console.log("404 ->", JSON.stringify(await l45toResult(l45mockResponse(404, { message: "Not Found" }))));
console.log("500 ->", JSON.stringify(await l45toResult(l45mockResponse(500, { message: "Boom" }))));

// The 404 case is the one to look at: the body parses perfectly well as JSON. Without the
// `ok` check you would return { message: "Not Found" } as if it were your users.

Note what the 404 line proves. There is nothing malformed about an error response — it is
valid JSON, it parses, and it would sail into your state unnoticed.

### Exercise

**Part 1 — in the notebook.** Extend `l45toResult` without breaking its contract.

1. A 204 response means "success, no content" and has **no body at all** — calling `.json()`
   on it throws. Handle it: a 204 should return `{ ok: true, data: null }` and must not call
   `.json()`.
2. A response can be successful and still contain something you cannot use. Add a check: if
   the parsed data is not an array, return
   `{ ok: false, error: "Expected a list" }`.
3. Prove all of it. Log the result for a 200 with an array, a 200 with an object, a 204, and
   a 404, and check each one is what you expect.

**Part 2 — in the playground.** Point `playground/src/App.jsx` at `./experiments/20-fetch.jsx`
and open the console **and** the Network tab.

1. Reload. Write down the order of the `render` and `request` lines. Which happens first, and
   why can it not be the other way round?
2. Click **break the URL (404)**. Read the console. Then open the Network tab and look at the
   request itself — what status did the browser report, and did `fetch` treat it as a failure?
3. In `20-fetch.jsx`, delete the two lines that check `response.ok`. Click **break the URL**
   again. What is now in `users`, what does the screen show, and why is no error displayed?
4. Put the check back.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

```jsx
function UserList({ url }) {
  const [users, setUsers] = useState([]);

  useEffect(() => {
    fetch(url)
      .then((response) => response.json())
      .then((data) => setUsers(data));
  }, [url]);

  return <ul>{users.map((user) => <li key={user.id}>{user.name}</li>)}</ul>;
}
```

This works on a good day. Find **four** separate problems with it, and for each say which
lesson covers the fix. Then answer:

- Which of the four would a user notice first on a slow connection?
- Which of the four produces a wrong screen rather than a crash — and why is that worse?

In [ ]:
// Your code here

## LESSON 46 — Loading, error, empty, success

The JavaScript course gave you three states to think about:

```text
    loading   →   success
              →   error
```

In a component there are **four**, and the one it left out is the one that causes the most
confusing screens.

### The fourth state is *empty*

A request can succeed perfectly and return nothing. No user matched the filter, the inbox has
no messages, the new account has no orders. Measured in the playground, asking for a user id
that does not exist:

```text
   request — .../users?id=99999
   ok — 0 users
   render — loading:false users:0 error:no
```

No error. Nothing failed. There is simply nothing there, and "nothing there" is a completely
different thing to say to a user than "still loading" or "something went wrong".

| state | what the user should understand |
|---|---|
| **loading** | it is coming, wait |
| **error** | it went wrong, here is what and what to do |
| **empty** | it worked, and there is nothing to show |
| **success** | here is your data |

Skip any one of those and the screen lies. An empty list rendered during loading says "you
have no messages" to somebody who has two hundred.

### The trap that hides *empty* behind *loading*

```jsx
const [users, setUsers] = useState([]);   // looks convenient
```

Starting at `[]` means `users.map(...)` never crashes, which is why people do it. It also
means **an empty array before the request and an empty array after the request are
indistinguishable**, so you cannot tell "not yet" from "none". The states collapse.

Two ways out, and both are fine:

```jsx
const [users, setUsers] = useState(null);     // null means "not yet", [] means "none"
```

or keep `[]` and track loading separately, which you need anyway:

```jsx
const [loading, setLoading] = useState(true);
```

The rule is that **the four states must be distinguishable from what you store**. If two of
them look identical in state, no amount of care in the JSX can separate them.

### Rendering them

Early returns (LESSON 18) keep this readable, and the order matters:

```jsx
if (loading) return <Spinner />;
if (error) return <ErrorMessage message={error} onRetry={reload} />;
if (users.length === 0) return <p>No users match that search.</p>;

return <ul>{users.map((user) => <li key={user.id}>{user.name}</li>)}</ul>;
```

Loading first, because while it is true nothing else is known. Error next, because an error
means the data is not trustworthy. Empty before success, because success is the only branch
that assumes there is something to show.

> Note what this is **not**. Deciding which of four branches to render is ordinary conditional
> rendering — no library, no state machine, no new API. The work is in being honest about which
> state you are in, not in the rendering.

### Key Notes

- There are **four** states, not three: loading, error, empty, success.
- *Empty* is a success. It needs its own message, and it is not "no data yet".
- Starting a list at `[]` makes empty and loading indistinguishable — store enough to tell
  them apart.
- Render them with early returns, in the order loading → error → empty → success.

### Example

**Runnable — plain JS, offline.** Which state you are in is a pure function of what you hold.
Extracting it makes the four cases testable, and makes it obvious when two of them collide.

In [ ]:
function l46statusOf({ loading, error, data }) {
  if (loading) return "loading";
  if (error) return "error";
  if (data === null || data === undefined) return "loading";   // not requested yet
  if (Array.isArray(data) && data.length === 0) return "empty";
  return "success";
}

const l46cases = [
  ["fresh, nothing requested ", { loading: false, error: null, data: null }],
  ["request in flight        ", { loading: true, error: null, data: null }],
  ["failed                   ", { loading: false, error: "500", data: null }],
  ["succeeded with nothing   ", { loading: false, error: null, data: [] }],
  ["succeeded with data      ", { loading: false, error: null, data: [{ id: 1 }] }],
];

for (const [label, state] of l46cases) {
  console.log(label, "->", l46statusOf(state));
}

// Now the trap, with data starting as [] instead of null:
console.log("");
console.log("start at [] , not loaded yet ->", l46statusOf({ loading: false, error: null, data: [] }));
console.log("start at [] , loaded, none   ->", l46statusOf({ loading: false, error: null, data: [] }));
console.log("^ identical inputs, so no function can tell these apart");

The last two lines are the argument for storing enough to distinguish the states. The function
is not at fault — the information simply is not there.

### Exercise

**Part 1 — in the notebook.** Extend `l46statusOf` to a real screen's needs.

1. Add a `stale` case: if `data` exists **and** `loading` is true, return `"refreshing"` — the
   situation where you already have results on screen and are fetching newer ones. Make sure
   the plain `loading` case still works when there is no data.
2. Add a guard: if `error` is set **and** `data` exists, you must decide what to show. Pick one
   and defend it in a comment — showing the stale data with a warning, or showing the error
   and hiding the data.
3. Log the result for six states: nothing yet, first load, failed first load, loaded-empty,
   loaded-with-data, and refreshing-with-data.

**Part 2 — in the playground**, in `20-fetch.jsx`, with the console open.

1. Click **a result with nothing in it**. Read the console and the screen. Is this an error?
   What does the screen currently say, and what *should* it say?
2. Add an empty-state message to the component: when the request has finished, there is no
   error, and the list is empty, show "No users match." Check it appears for the empty URL and
   not for the others.
3. Look at the initial value of `users`. It starts at `null`. Change it to `[]` and reload —
   which of the four states can you no longer distinguish, and where does that show up on
   screen?
4. Put it back to `null`.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

A support ticket says: *"The app told me I have no invoices. I definitely have invoices."*

Answer in comments:

1. Give **three** different bugs that could produce that exact complaint, each involving a
   different pair of the four states being confused.
2. For each, say what the user would have seen instead if the state had been handled properly.
3. Which of the three is impossible to diagnose from the screenshot the user sent, and why?
4. The developer's first instinct is to add a spinner. Explain why that fixes at most one of
   the three.

In [ ]:
// Your code here

## LESSON 47 — Fetching inside an Effect, properly

You now have every piece. This lesson puts them together and adds the one thing still missing.

### The complete shape

```jsx
useEffect(() => {
  const controller = new AbortController();
  let ignore = false;

  async function load() {
    setLoading(true);
    setError(null);

    try {
      const response = await fetch(`/api/users/${userId}`, { signal: controller.signal });
      if (!response.ok) throw new Error(`Request failed with status ${response.status}`);
      const data = await response.json();
      if (!ignore) setUsers(data);
    } catch (problem) {
      if (problem.name === "AbortError") return;      // we cancelled it; not an error
      if (!ignore) setError(problem.message);
    } finally {
      if (!ignore) setLoading(false);
    }
  }

  load();

  return () => {
    ignore = true;
    controller.abort();
  };
}, [userId]);
```

Everything in it is something you have already learned, except one line:

| line | lesson |
|---|---|
| the Effect at all | L39 — caused by rendering |
| `[userId]` | L40 — refetch when the thing being fetched changes |
| the returned cleanup | L41 |
| the inner `async function` | L43 — the callback cannot be async |
| `ignore` | L43 — stop a stale response changing state |
| `!response.ok` | the JavaScript course, L45 |
| `AbortController` | **new** |

### Deps: the URL is a dependency

`[userId]` is not decoration. Change the id and you want a different request; that is exactly
what a dependency array is for (LESSON 40). Get it wrong in the two familiar ways and you get
the two familiar bugs:

- `[]` — the component fetches user 1 and then shows user 1's data forever, however many times
  the prop changes. The screen and the data drift apart.
- an object dependency (`[{ userId }]`) — a new object every render, so a new request after
  every commit, forever.

### `ignore` and `abort` do different jobs

This is the distinction worth getting right, because people reach for one and think they have
both.

| | what it does | what it does not do |
|---|---|---|
| `ignore` | stops a stale response **changing your state** | the request still runs, still costs bandwidth, still arrives |
| `controller.abort()` | stops **the request itself** | does not retroactively undo anything already applied |

React's position on the first is worth keeping:

> You can't "undo" a network request that already happened, but your cleanup function should
> ensure that the fetch that's *not relevant anymore* does not keep affecting your application.

Use both. `abort` saves the work; `ignore` guarantees correctness even in the gap between
deciding to abort and the abort taking effect.

### The trap: an aborted request throws

This is the part nobody warns you about. Aborting makes the fetch promise **reject**, and that
rejection lands in your `catch` like any other failure. Measured, three presses in a row:

```text
   forced — starting a request and aborting it in the same tick
   forced — caught AbortError: signal is aborted without reason
```

If your `catch` does `setError(problem.message)` unconditionally, then every time the user
types another character, navigates away, or the component re-renders with a new id, they get
an error message on screen — caused by **your own cleanup**. The app appears to fail constantly
and only while things are working normally.

Hence the first line of the catch block:

```js
if (problem.name === "AbortError") return;
```

Check the name, not the message: the message text differs between browsers, the name does not.

### Key Notes

- The full pattern is deps + cleanup + inner async + `ignore` + `ok` check + abort. Only the
  last is new.
- The thing you are fetching **by** belongs in the dependency array.
- `ignore` protects your state; `abort` stops the request. They are not alternatives.
- An aborted request **rejects** — detect `AbortError` by name and return, or your own cleanup
  will show errors to the user.

### Example

**In the playground.** No cell — an abort is a real network event and a mock would prove
nothing about it.

Point `playground/src/App.jsx` at `./experiments/21-abort.jsx`. Press **abort immediately**
and read the two lines; the abort is issued in the same tick as the request, so the rejection
is guaranteed rather than a matter of luck. Then press **reload** and watch the cleanup abort
the previous request before the new one starts.

### Exercise

**In the playground**, in `21-abort.jsx`.

1. Press **abort immediately** three times. Write down the exact `name` and `message` you get.
   Which of the two would you write a condition against, and why?
2. Find the line `if (problem.name === "AbortError") return;` in the Effect and delete it.
   Press **reload** several times quickly. What appears in *last error seen*, and why is it
   misleading?
3. Put it back. Now delete `controller.abort()` from the cleanup but keep `ignore = true`.
   Press **reload** several times and watch the Network tab. What still happens that you have
   not stopped, and what is protected anyway?
4. In a comment: give one situation where `ignore` alone is genuinely enough, and one where
   leaving out `abort` would matter.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

```jsx
// A
useEffect(() => {
  const controller = new AbortController();
  fetch(url, { signal: controller.signal })
    .then((r) => r.json())
    .then(setData)
    .catch((e) => setError(e.message));
  return () => controller.abort();
}, [url]);

// B
useEffect(() => {
  let ignore = false;
  async function load() {
    const r = await fetch(`/api/users/${userId}`);
    if (!ignore) setUser(await r.json());
  }
  load();
  return () => { ignore = true; };
}, []);

// C
useEffect(() => {
  const controller = new AbortController();
  async function load() {
    try {
      const r = await fetch(url, { signal: controller.signal });
      if (!r.ok) throw new Error(r.status);
      setData(await r.json());
    } catch (e) {
      if (e.name !== "AbortError") setError(e.message);
    }
  }
  load();
  return () => controller.abort();
}, [url]);
```

For each: list every defect, and say whether the user would see a wrong screen, a spurious
error, or nothing at all. Then answer:

- **C** has no `ignore` flag and is still substantially correct. Explain when that is safe and
  what single change would make it unsafe.
- Which of the three would pass a code review by someone who had only read LESSON 43?

In [ ]:
// Your code here

## LESSON 48 — A small services layer

The JavaScript course already gave you the rule, and even told you where it was going:

> **`services/`** — knows URLs and JSON. Knows nothing about the page.
> the UI — knows the page. Contains **no URL**.
> …This is the `src/services/` folder you will build again in the final project, and the same
> split React projects use.

This lesson is that promise being kept, and it is short because the idea is not new — only the
React half is.

### What each side owns

```text
component  ──calls──▶  service  ──calls──▶  fetch  ──▶  the network
   ▲                      │
   └──── plain result ────┘
```

| | knows | never knows |
|---|---|---|
| **service** | the URL, the shape of the response, how to turn a failure into a result | that React exists, that state exists, what a component is |
| **component** | the four states, what to render for each | the URL, the response shape, HTTP status codes |

The service returns a **plain result** — not a Promise of a component, not something that sets
state. Exactly the shape from LESSON 45:

```js
// services/users.js
const BASE = "https://jsonplaceholder.typicode.com";

export async function getUsers({ signal } = {}) {
  const response = await fetch(`${BASE}/users`, { signal });

  if (!response.ok) {
    return { ok: false, error: `Request failed with status ${response.status}` };
  }

  return { ok: true, data: await response.json() };
}
```

And the component stays about the screen:

```jsx
useEffect(() => {
  const controller = new AbortController();
  let ignore = false;

  async function load() {
    setLoading(true);
    const result = await getUsers({ signal: controller.signal });
    if (ignore) return;

    if (result.ok) {
      setUsers(result.data);
      setError(null);
    } else {
      setError(result.error);
    }
    setLoading(false);
  }

  load();
  return () => { ignore = true; controller.abort(); };
}, []);
```

There is no URL in that component and no `useState` in that service. That is the entire rule.

### Why it is worth the extra file

Three reasons, in the order you will actually feel them:

1. **One place to change.** The API moves, a path changes, a header is added — one file.
2. **It is testable without React.** `getUsers` is an async function that returns an object.
   You can call it from a notebook cell, which is exactly what the exercise does.
3. **The component gets shorter and more honest.** It stops being about HTTP and goes back to
   being about what the user sees.

### Keep it small

A services layer is one file per resource with a few exported functions. That is all.

**Do not** add a repository, a generic `ApiClient` class, a `BaseService` to inherit from, a
request/response interceptor stack, or a folder of abstractions between the component and
`fetch`. Every one of those is a real pattern somewhere, and every one of them is heavier than
a course-sized project can justify. Two files and a clear rule beat an architecture.

> Passing `signal` through is the one piece of plumbing worth having from the start. Without
> it the component cannot cancel, and you would have to change every service function later to
> add it.

### Key Notes

- The **service** owns the URL and the response shape. The **component** owns the screen.
- A service returns a plain result object — it never touches state.
- One file per resource, a few functions. No repositories, clients or base classes.
- Thread `signal` through so the component can still abort.

### Example

**Runnable — plain JS, offline.** A service is an ordinary async function, which means it can
be tested without React *and* without the network — you hand it a fake `fetch`. That is not a
trick for the notebook; it is the reason this split is worth having.

In [ ]:
// services/users.js, with fetch injected so it can run anywhere.
function l48makeUserService(fetchImpl) {
  const BASE = "https://example.test";

  return {
    async getUsers({ signal } = {}) {
      const response = await fetchImpl(`${BASE}/users`, { signal });
      if (!response.ok) {
        return { ok: false, error: `Request failed with status ${response.status}` };
      }
      return { ok: true, data: await response.json() };
    },
  };
}

// Fake responses. No network, no React.
const l48ok = async () => ({ ok: true, status: 200, json: async () => [{ id: 1, name: "Ada" }] });
const l48notFound = async () => ({ ok: false, status: 404, json: async () => ({}) });
const l48serverError = async () => ({ ok: false, status: 500, json: async () => ({}) });

console.log("200 ->", JSON.stringify(await l48makeUserService(l48ok).getUsers()));
console.log("404 ->", JSON.stringify(await l48makeUserService(l48notFound).getUsers()));
console.log("500 ->", JSON.stringify(await l48makeUserService(l48serverError).getUsers()));

// Every branch of the service checked, in a notebook, offline. The component that uses it
// needs none of this - it only ever sees { ok, data } or { ok, error }.

That is point 2 from the lesson, demonstrated rather than claimed: the whole service was
exercised without a browser, without React and without a network.

### Exercise

**Part 1 — in the notebook.** Grow the service without letting React leak into it.

1. Add `getUser(id, { signal })` which requests `${BASE}/users/${id}` and returns the same
   result shape. A 404 here means "no such user" — return
   `{ ok: false, error: "User not found", notFound: true }` so the component can tell that
   apart from a server failure.
2. Add `searchUsers(query, { signal })` which requests `${BASE}/users?q=` plus the query,
   **encoded properly**. Think about what happens if the query contains a space or an
   ampersand.
3. Test all of it with fake fetches: a found user, a missing user, a 500, and a search whose
   query needs encoding. Log the URL each fake fetch received so you can prove the encoding
   worked.

**Part 2 — in your project.** You will build this for real in Mini-project 3. For now, sketch
it in comments: list the files you would create for an app that shows a list of employees and
one employee's detail page, and say for each file what it knows and what it must not know.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

Each of these puts something on the wrong side of the line. For each, say what has leaked and
which direction it went.

```js
// A
export async function getUsers(setUsers, setError) {
  const response = await fetch(`${BASE}/users`);
  if (!response.ok) return setError("failed");
  setUsers(await response.json());
}

// B
function UserList() {
  useEffect(() => {
    fetch("https://api.example.com/v2/users?active=true&limit=50")
      .then((r) => r.json())
      .then(setUsers);
  }, []);
}

// C
export async function getUsers() {
  const response = await fetch(`${BASE}/users`);
  return response;
}

// D
export async function getUsers() {
  const response = await fetch(`${BASE}/users`);
  const users = await response.json();
  return users.map((u) => <li key={u.id}>{u.name}</li>);
}
```

Then answer: **C** looks harmless — it returns the response and lets the caller decide. Say
what it forces every caller to duplicate, and why that is the failure the services layer exists
to prevent.

In [ ]:
// Your code here